# Séance 3 — Cryptanalyse de Vigenère (Kasiski, Friedman) et code de VIC

**Cours de référence** : P. Boyer, *Cryptographie 2013*, fin du chapitre 2
(« Cryptanalyse », « Code de VIC »).

## Objectifs de la séance

1. trouver la **période** d'un message de Vigenère sans aucune indication, par deux
   méthodes concurrentes :
   * la méthode de **Kasiski** (1853) : répétitions et distances multiples de la période ;
   * l'**indice de coïncidence** de Friedman (années 1920) : découpage en colonnes ;
2. casser complètement les deux dépêches interceptées ;
3. découvrir le **code de VIC** (service secret britannique, 1914) dans sa version
   simplifiée : addition de clef, transposition, damier de 10 × 10.

## Ce que vous rendez

Votre fichier `tp3_vigenere_vic.py` complété (quatorze fonctions) et les réponses aux
questions de rapport.

    python3 test_tp3_vigenere_vic.py


## 1. Rappels : deux façons de trouver la période

### Méthode de Kasiski

Si un groupe de lettres apparaît deux fois dans le texte clair, il apparaîtra deux fois
dans le texte chiffré, décalé du **même décalage** (car sous la même suite de lettres de
clef). La distance entre les deux répétitions est donc **un multiple de la période**.
On cherche les répétitions, on calcule les distances, et l'on regarde quels diviseurs
reviennent le plus souvent.

### Indice de coïncidence (Friedman)

L'indice de coïncidence d'un texte est la probabilité que deux lettres prises au hasard
y soient égales :

* texte français : environ **0,075** ;
* texte bien mélangé (26 lettres presque équiprobables) : environ **0,038**.

Si l'on découpe le texte chiffré en $p$ colonnes et que $p$ est la période, chaque colonne
est un César du français, donc son indice vaut environ 0,075. Sinon, la moyenne retombe
vers 0,04. **Attention** : les multiples de la période (2p, 3p...) donnent eux aussi un
indice élevé : il faut prendre la plus petite période qui « fonctionne ».


In [ ]:
import os, sys, subprocess

def trouver_racine(depart):
    """Remonte les dossiers jusqu'à trouver la racine du dépôt."""
    while True:
        if os.path.exists(os.path.join(depart, "outils", "reference.py")):
            return depart
        parent = os.path.dirname(depart)
        if parent == depart:
            raise RuntimeError("racine du dépôt introuvable")
        depart = parent

RACINE = trouver_racine(os.getcwd())
if RACINE not in sys.path:
    sys.path.insert(0, RACINE)

def trouver(nom):
    """Retrouve un fichier du dépôt (son dossier de séance, peu importe où l'on est)."""
    for base, _, fichiers in os.walk(RACINE):
        if nom in fichiers:
            return os.path.join(base, nom)
    raise FileNotFoundError(nom)

DOSSIER = os.path.dirname(trouver("test_tp3_vigenere_vic.py"))

def verifier():
    """Lance les épreuves automatiques de la séance et affiche le bilan."""
    resultat = subprocess.run([sys.executable, "test_tp3_vigenere_vic.py"], cwd=DOSSIER,
                              capture_output=True, text=True)
    print(resultat.stdout.strip()[-2500:])

def executer(consigne, fonction):
    """Exécute une cellule d'exercice, avec un rappel si votre fonction manque encore."""
    try:
        return fonction()
    except NotImplementedError as erreur:
        print("Cette cellule attend une fonction de votre fichier :", consigne)
        print("   ->", erreur)
        print("   Écrivez la fonction, enregistrez le fichier .py, puis relancez la cellule.")
        return None

def charger():
    """Recharge votre fichier de TP (après l'avoir modifié, relancez cette cellule)."""
    import importlib
    if DOSSIER not in sys.path:
        sys.path.insert(0, DOSSIER)
    if "tp3_vigenere_vic" in sys.modules:
        return importlib.reload(sys.modules["tp3_vigenere_vic"])
    import importlib.util
    spec = importlib.util.spec_from_file_location("tp3_vigenere_vic", os.path.join(DOSSIER, "tp3_vigenere_vic.py"))
    module = importlib.util.module_from_spec(spec)
    sys.modules["tp3_vigenere_vic"] = module
    spec.loader.exec_module(module)
    return module

print("Racine du dépôt :", RACINE)
print("Dossier de la séance :", DOSSIER)


## 2. Mise en œuvre (partie A)

Écrivez `indice_coincidence_moyen`, `indices_par_periode` et `periode_probable` dans
votre fichier, puis exécutez la cellule ci-dessous : elle affiche le **tableau des
indices** du premier message intercepté. C'est le cœur de la méthode de Friedman.


In [ ]:
tp = charger()

def _exercice():
    indices = tp.indices_par_periode(tp.MESSAGE_1, 15)
    meilleur = max(indices.values())
    periode_choisie = tp.periode_probable(tp.MESSAGE_1)
    print("période :  indice   (barre relative au meilleur indice)")
    for periode in sorted(indices):
        valeur = indices[periode]
        barre = "#" * int(round(40 * valeur / meilleur))
        marque = ""
        if periode_choisie and periode % periode_choisie == 0:
            marque = "   <-- multiple de la période retenue"
        print(f"  {periode:2d}    : {valeur:.4f}  {barre}{marque}")
    print()
    print("Période retenue :", periode_choisie)

executer("A.1 à A.3 (indices de coïncidence)", _exercice)
verifier()

## 3. La méthode de Kasiski (partie B)

Écrivez `grammes_repetes`, `kasiski_distances` et `diviseurs`, puis observez les
répétitions du message intercepté. La cellule ci-dessous répète exactement la démarche
de Kasiski : pour chaque groupe répété, les distances ; pour chaque distance, ses
diviseurs ; et l'on compte le support de chaque période possible.


In [ ]:
tp = charger()

def _exercice():
    chiffre = tp.MESSAGE_1
    for taille in (3, 4, 5):
        groupes = tp.grammes_repetes(chiffre, taille)[:8]
        print(f"--- groupes de {taille} lettres répétés (les 8 premiers) : {groupes}")
    print()
    support = {p: 0 for p in range(2, 16)}
    for taille in (4, 5):
        for groupe in tp.grammes_repetes(chiffre, taille):
            for distance in tp.kasiski_distances(chiffre, groupe):
                for periode in range(2, 16):
                    if distance % periode == 0:
                        support[periode] += 1
    print("période : nombre de distances qu'elle divise")
    for periode in sorted(support, key=lambda p: -support[p])[:8]:
        print(f"  {periode:2d} : {support[periode]}")

executer("B.1 à B.3 (Kasiski)", _exercice)

**Discussion** : les petits nombres (2, 3, 4) divisent beaucoup de distances, même
quand ils ne sont pas la vraie période. C'est le défaut bien connu de la méthode de
Kasiski : elle donne des **candidats**, pas une réponse. L'indice de coïncidence
tranche : ici, seules 4 et ses multiples donnent un indice proche de 0,075.

C'est exactement ce que font les cryptanalystes : **croiser deux indices faibles**
(Kasiski + Friedman) pour obtenir une conclusion solide.


## 4. Casser les deux dépêches (partie C)

Écrivez `forcer_cesar_sur_colonne` et `forcer_vigenere`, puis attaquez les deux messages.
Cette fois, aucune période ne vous est donnée : c'est à vous de la trouver (partie A).


In [ ]:
tp = charger()

def _exercice():
    for nom, message in (("Dépêche 1", tp.MESSAGE_1), ("Dépêche 2", tp.MESSAGE_2)):
        periode = tp.periode_probable(message)
        cle, clair = tp.forcer_vigenere(message, periode)
        print("=" * 74)
        print(f"{nom} : {len(message)} lettres — période trouvée {periode} — clef {cle}")
        print(clair[:220], "...")

executer("A.3 puis C.1, C.2 (déchiffrement complet)", _exercice)
verifier()

## 5. Le code de VIC (partie D)

Le VIC est le chiffre du service secret britannique pendant la Première Guerre mondiale.
Le cours en décrit quatre étapes :

1. on écrit la clef au-dessus du message (elle est écrite **deux fois**, ce qui permet au
   destinataire de la retrouver sans la transmettre) ;
2. on additionne la clef au message, chiffre à chiffre, sans retenue ;
3. on brouille l'ordre des caractères par une transposition ;
4. on transforme chaque chiffre en lettre à l'aide d'un **damier** de 10 × 10 cases.

Le VIC n'a été cassé qu'en 1914 par les Britanniques, et il n'est devenu public que
dans les années 1930. Sa faiblesse principale : la clef était écrite dans le message.

### Notre version simplifiée

Pour que tout soit calculable et vérifiable, nous remplaçons les tirages au hasard du
vrai VIC par des choix déterministes : le « chiffre indicateur » est la **position du
caractère dans son bloc de 10**, et la clef est fournie directement. Tout est détaillé
dans `outils/reference.py` (section 5).


In [ ]:
from outils.reference import afficher_tableau_vic, ALPHABET_VIC, vic_chiffrer
print("Alphabet du damier (10 lettres) :", ALPHABET_VIC)
print()
print(afficher_tableau_vic())
print()
# Vérification de la propriété essentielle : carré latin
for ligne in range(10):
    assert sorted(__import__("outils.reference", fromlist=["x"]).TABLEAU_VIC[ligne]) == sorted(ALPHABET_VIC)
print("Chaque ligne contient les 10 lettres du damier, une fois et une seule.")

Écrivez maintenant `caractere_vic`, `ligne_du_damier`, `chiffre_du_damier` et
`dechiffrer_message_vic`, puis déchiffrez le message VIC intercepté (sa clef vous est
donnée : `CLE_VIC`).


In [ ]:
tp = charger()

def _exercice():
    print("Message chiffré   :", tp.MESSAGE_VIC)
    print("Message déchiffré :", tp.dechiffrer_message_vic(tp.MESSAGE_VIC, tp.CLE_VIC))

executer("D.1 à D.4 (VIC simplifié)", _exercice)
verifier()

## 6. Questions de rapport

1. **Kasiski.** Démontrez la propriété clé : si un même groupe de lettres apparaît deux
   fois dans le texte clair et que la distance entre les deux occurrences n'est pas un
   multiple de la période, que faudrait-il pour que le texte chiffré contienne encore le
   même groupe répété ? Concluez que les distances sont presque toujours multiples de la
   période.
2. Pourquoi les **multiples** de la vraie période donnent-ils aussi un indice de
   coïncidence élevé ? Comment votre fonction `periode_probable` résout-elle le problème ?
3. Comparez les méthodes de Kasiski et de Friedman : laquelle coûte le plus cher en
   calcul ? Laquelle est la plus fiable sur un message de 200 lettres ?
4. **VIC.** Expliquez le rôle de chacune des quatre étapes. Que se passerait-il si l'on
   retirait l'étape 3 (la transposition) ? Et l'étape 2 (l'addition de clef) ?
5. Le damier du VIC est public. En quoi cela illustre-t-il le principe de Kerckhoffs ?
6. Le VIC a été cassé parce que la même clef de message servait deux fois, produisant des
   répétitions exploitables. Rapprochez cela de la leçon de la question 4 de la séance 2.
7. Quel lien faites-vous entre l'amélioration progressive des chiffres (César → Vigenère
   → VIC → Enigma) et les moyens de calcul disponibles à chaque époque ?


## 7. Pour aller plus loin (facultatif)

* **Attaque avec mot probable** : supposons qu'une dépêche contienne le mot « COMMANDANT ».
  Pour chaque position possible de ce mot, on essaie : le décalage obtenu pour la position
  $i$ donne la lettre de clef $i \bmod \ell$. Faites-le à la main sur `MESSAGE_1` et
  retrouvez ainsi la période par recoupement (c'est la méthode utilisée contre Enigma,
  et adaptée ici au Vigenère).
* **Indice de coïncidence mutuel** : au lieu de comparer des colonnes au français, on
  compare deux textes chiffrés avec la même clef (méthode de Friedman pour retrouver la
  période sans langue de référence). C'est essentiel pour les messages sans espaces.


In [ ]:
# Attaque par mot probable sur MESSAGE_1
from outils.reference import ALPHABET
chiffre = tp.MESSAGE_1
mot = "COMMANDANT"
for position in range(0, 60):
    if position + len(mot) > len(chiffre):
        break
    candidat = chiffre[position:position + len(mot)]
    decalages = [(ALPHABET.index(c) - ALPHABET.index(p)) % 26 for c, p in zip(candidat, mot)]
    print(f"position {position:3d} : décalages {decalages}")
print()
print("Pour la bonne position, ces décalages doivent se répéter avec la période de la clef.")